In [1]:
import pandas as pd
from langchain_core.documents import Document

# ubah path jika perlu
df = pd.read_excel("chatbot_its_dataset2.xlsx")
df.head()

# Tampilkan kolom agar tahu nama kolom persis
print("Columns:", df.columns.tolist())

# safe mapping in case columns have different exact names:
q_col = df.columns[0]
i_col = df.columns[1]
a_col = df.columns[2]

print("Using columns: ", q_col, " -> ", a_col)

# Create knowledge chunks: store both Q and A as context
documents = []
metas = []
for j, row in df.iterrows():
    q = str(row[q_col])
    i = str(row[i_col])
    a = str(row[a_col])
    text = f"FAQ Question: {q}\nReference Answer: {a}"
    documents.append(Document(page_content=text, metadata={"source_row": int(j)}))


Columns: ['question', 'intent', 'response']
Using columns:  question  ->  response


In [2]:
from langchain_ollama import OllamaLLM

# jika Ollama serve default di localhost:11434, wrapper akan memakai itu
llm = OllamaLLM(model="llama3.1")  # ganti model sesuai yang sudah kamu pull


C:\Users\raras\anaconda3\envs\openllm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import pickle
from pathlib import Path

# config
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_DIR = "./stored"
INDEX_PATH = f"{INDEX_DIR}/faiss_index.idx"
CHUNKS_PATH = f"{INDEX_DIR}/chunks.pkl"
METAS_PATH = f"{INDEX_DIR}/metas.pkl"

Path(INDEX_DIR).mkdir(parents=True, exist_ok=True)

# embed function (batch)
embed_model = SentenceTransformer(EMB_MODEL)

texts = [doc.page_content for doc in documents]
print("Total chunks:", len(texts))

# compute embeddings (batch)
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

# build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
faiss.write_index(index, INDEX_PATH)

# save chunks & metas
with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(texts, f)
with open(METAS_PATH, "wb") as f:
    pickle.dump([doc.metadata for doc in documents], f)

print("Saved FAISS index, chunks, metas in", INDEX_DIR)


Total chunks: 25


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.39it/s]

Saved FAISS index, chunks, metas in ./stored


In [4]:
def generate_from_llm(prompt, max_tokens=256):
    return llm.invoke(prompt)

In [5]:
import pickle

# load index & chunks
index = faiss.read_index(INDEX_PATH)
with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)
with open(METAS_PATH, "rb") as f:
    metas = pickle.load(f)

def retrieve(query, k=3):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_emb, k)    # D = L2 distance
    results = []
    for idx in I[0]:
        results.append({"text": chunks[idx], "metadata": metas[idx]})
    return results, D[0]


In [6]:
from textwrap import shorten

SYSTEM_INSTR = (
    "Kamu adalah asisten resmi yang menjawab pertanyaan terkait proses akademik ITS. "
    "Berikan jawaban terstruktur dengan format: "
    "1. Penjelasan ringkas "
    "2. Detail persyaratan / prosedur "
    "3. Catatan tambahan / tips penting "

    "Gunakan konteks sebagai sumber data faktual, tetapi jangan menyalin kalimat secara langsung."
    "Susun ulang dengan bahasa informatif dan lebih lengkap. "
    "Jika konteks tidak cukup untuk menjawab, bilang 'Maaf, informasi tidak tersedia dalam sumber.'"
)

def build_prompt(user_q, retrieved, prompt_tambahan):
    # build a context string listing retrieved items with small excerpts
    context_blocks = []
    for i, r in enumerate(retrieved, 1):
        # limit length to avoid too long prompt
        excerpt = shorten(r["text"], width=600, placeholder="...")
        context_blocks.append(f"[{i}] {excerpt}")
    context = "\n\n".join(context_blocks)
    prompt = (
        SYSTEM_INSTR + "\n\n" +
        f"Context (reference QA pairs):\n{context}\n\n"
        f"Pertanyaan pengguna: {user_q}\n\n" + prompt_tambahan
    )
    # print (prompt)
    return prompt


In [11]:
### LANGGRAPH

from typing import TypedDict
from langgraph.graph import StateGraph

class State(TypedDict):
    query: str
    answer: str
    distance: float
    k: int
    retrieved: str
    prompt: str

def node_retriever(state):
    query = state["query"]
    k = state["k"]
    results, distances = retrieve(query, k)

    # Ambil dokumen dengan distance terkecil (paling mirip)
    best_idx = distances.argmin()
    best_doc = results[best_idx]["text"]
    best_dist = float(distances[best_idx])

    return {"retrieved": results, "distance": best_dist}
    
# --- NODE: router ---
def router(state: State):
    dist = state["distance"]
    threshold = 1.2
    if dist <= threshold:
        return {"next_node": "node_rag_answer"}
    else:
        return {"next_node": "node_fallback"}

def node_fallback(state: State):
    ans = "Mohon maaf saya tidak bisa menjawab pertanyaan ini karena di luar konteks yang saya pahami."
    retrieved = ""
    prompt = state["query"]
    return {"answer": ans, "retrieved": retrieved, "prompt": prompt}

def node_rag_answer(state: State):
    q = state["query"]
    k = state["k"]
    retrieved = state["retrieved"]

    prompt_tambahan = ("Gunakan konteks di atas sekaligus lakukan pencarian di web terkait pertanyaan user untuk menyusun jawaban baru."
    "Jawaban juga harus mencantumkan link URL valid dari website terkait yang menjadi sumber jawaban sebagai bagian dari catatan tambahan / tips penting.")
    prompt = build_prompt(q, retrieved, prompt_tambahan)

    llm_out = generate_from_llm(prompt)
    # llm_out may be LangChain object/string; convert to string if needed
    ans = llm_out if isinstance(llm_out, str) else str(llm_out)
    return {"answer": ans, "retrieved": retrieved, "prompt": prompt}

# --- BANGUN GRAPH ---
graph = StateGraph(State)

graph.add_node("node_retriever", node_retriever)
graph.add_node("router", router)
graph.add_node("node_rag_answer", node_rag_answer)
graph.add_node("node_fallback", node_fallback)

graph.set_entry_point("node_retriever")
graph.add_edge("node_retriever", "router")

# mapping edge berdasarkan nilai "route"
graph.add_conditional_edges(
    "router",
    lambda state: state["next_node"],
    {
        "node_rag_answer": "node_rag_answer",
        "node_fallback": "node_fallback"
    }
)

app = graph.compile()


In [12]:
print("RAG chatbot siap. Ketik 'exit' untuk keluar.")
while True:
    q = input("Tanya: ").strip()
    if q.lower() in ("exit","quit"):
        break
    res = app.invoke({"query": q, "k": 3})
    print("\nDistance: ", res["distance"])
    print("\nJawaban:\n", res["answer"])
    print("\nSumber referensi:")
    for i, r in enumerate(res["retrieved"],1):
        print(f"[{i}] row={r['metadata'].get('source_row')}  snippet: {r['text'][:200]}...")
    print("\n---\n")


RAG chatbot siap. Ketik 'exit' untuk keluar.


Tanya:  Bagaimana cara membayar UKT di ITS?



Distance:  0.7032675743103027

Jawaban:
 1. **Penjelasan ringkas**: Membayar UKT (Uang Kuliah Tunggal) di ITS dapat dilakukan melalui laman myITS Finance, setelah melihat tagihan yang telah dikeluarkan untuk Anda.

2. **Detail persyaratan/prosedur**:
   - Pastikan Anda memiliki akses ke myITS dan memiliki akun aktif.
   - Periksa tagihan UKT di menu yang relevan untuk memastikan tidak ada kesalahan dalam jumlah atau waktu pembayaran.
   - Pilih metode pembayaran yang tersedia, yaitu transfer bank atau kartu kredit (jika tersedia).
   - Ikuti instruksi pembayaran yang telah disediakan, termasuk nomor rekening dan kode pembayaran.
   - Lakukan pembayaran sebelum tanggal batas untuk menghindari denda.

3. **Catatan tambahan/tips penting**:
   - Pastikan Anda memiliki salinan tagihan untuk keperluan referensi.
   - Informasi tentang metode pembayaran dapat berubah, jadi periksa laman myITS Finance secara teratur.
   - Jika Anda mengalami kesulitan dalam melakukan pembayaran atau memiliki 

Tanya:  exit
